# Worked solutions: 02 Great Expectations exercises

Run top to bottom (about 1 minute). Everything here works on temporary copies: the
GX project is copied to a temp folder and the test data is generated into a temp folder.
Nothing in your `gx/` folder, `build_suite.py`, `validate.py` or `data/` is changed.

Run it with the lesson's Python environment, from inside `02-great-expectations/`.

In [1]:
# Setup: temporary copy of the GX project + generated test data
import json, os, shutil, subprocess, sys, tempfile, warnings
os.environ["TQDM_DISABLE"] = "1"      # hide progress bars
from pathlib import Path
warnings.filterwarnings("ignore")

import pandas as pd
import great_expectations as gx
import great_expectations.expectations as gxe

LESSON = Path.cwd()                       # the 02-great-expectations folder
sys.path.insert(0, str(LESSON))
import build_suite                        # the repo's own rules (importing runs nothing)
import validate                           # the repo's own gate (validate_file)

TMP = Path(tempfile.mkdtemp(prefix="gx_solutions_"))
shutil.copytree(LESSON / "gx", TMP / "gx", ignore=shutil.ignore_patterns("uncommitted"))
ctx = gx.get_context(mode="file", project_root_dir=TMP)   # works on the copy
from great_expectations.data_context.types.base import ProgressBarsConfig
ctx.variables.progress_bars = ProgressBarsConfig(globally=False, metric_calculations=False)  # quiet output
gate = ctx.checkpoints.get("emr_gate")                    # same checkpoint, temp project
batch_def = ctx.data_sources.get("emr").get_asset("visits").get_batch_definition("monthly_extract")

SIM = (LESSON / ".." / "00-data" / "simulate_emr.py").resolve()
for scenario in ["baseline", "messy_export"]:
    subprocess.run([sys.executable, str(SIM), "--start", "2025-01", "--scenario", scenario,
                    "--out", str(TMP / "data" / scenario)], check=True, capture_output=True)
baseline = pd.read_csv(TMP / "data/baseline/emr_2025-01.csv")
messy = pd.read_csv(TMP / "data/messy_export/emr_2025-01.csv")

def show(rep):
    """Print one validate_file() report the way validate.py does."""
    status = "FAIL" if rep["critical_failures"] else ("WARN" if rep["warnings"] else "PASS")
    print(f"{status}  {Path(rep['file']).parent.name}/{Path(rep['file']).name}  "
          f"({rep['rows']} rows, {rep['evaluated']} expectations)")
    for item in rep["critical_failures"] + rep["warnings"]:
        print(f"   [{item['severity']}] {item['expectation']}({item['column']}): {item['detail']}")

print("suite on disk:", len(ctx.suites.get("emr_monthly_extract").expectations), "expectations")
print("baseline:", baseline.shape, " messy_export:", messy.shape)

suite on disk: 42 expectations
baseline: (2000, 23)  messy_export: (2015, 23)


## Exercise 1

Generate a month where only one region reports
(hint: filter a baseline file to `region == "Oromia"` with pandas). Which
expectations fire, and at what severity? Should "only 5 facilities
reported" block training?

In [2]:
# Keep only Oromia rows of a clean baseline month and save as its own file
oromia = baseline[baseline["region"] == "Oromia"]
path = TMP / "data/oromia_only/emr_2025-01.csv"
path.parent.mkdir(parents=True, exist_ok=True)
oromia.to_csv(path, index=False)
print(len(oromia), "rows,", oromia["facility_id"].nunique(), "facilities")

633 rows, 5 facilities


In [3]:
# Validate it with the repo's own gate logic
rep = validate.validate_file(gate, path)
show(rep)

WARN  oromia_only/emr_2025-01.csv  (633 rows, 42 expectations)
   [warning] expect_column_unique_value_count_to_be_between(facility_id): observed 5


In [4]:
# What would the gate do? Same rule as validate.py: block only on critical failures
print("BLOCKED" if rep["critical_failures"] else "OK: no critical failures (training would go ahead)")

OK: no critical failures (training would go ahead)


**Answer**

Only one expectation fires: `expect_column_unique_value_count_to_be_between(facility_id)`, with severity **warning** (observed 5 facilities, the suite wants at least 20). The 633 Oromia rows are inside the 500 to 5000 bound, every region, code and range check is valid, and the gate says `OK`.

Should it block training? **No.** The data is not broken, it is just narrow. A warning is the right level: a human should look (did the other regions really not report, or did an export fail?), but blocking would stop a model trained on valid rows. The real risk is that a model trained or scored on one region is a different population from the one it was built on. That is a drift question (Evidently), not a data-quality one. If you want a hard stop, the better rule is a critical check that all 8 regions are present, which is a statement about completeness rather than a head count.

## Exercise 2

The `messy_export` duplicates are exact copies of real visits. Write an
expectation that would also catch the same child seen twice on the same day
at the same facility under different `visit_id`s
(hint: `ExpectCompoundColumnsToBeUnique`).

In [5]:
# The new expectation: one child, one day, one facility = one visit
same_visit_once = gxe.ExpectCompoundColumnsToBeUnique(
    column_list=["child_id", "visit_date", "facility_id"])

def run(expectation, df):
    batch = batch_def.get_batch(batch_parameters={"dataframe": df})
    r = batch.validate(expectation)
    return r.success, r.result["unexpected_count"], r.result.get("partial_unexpected_list", [])[:2]

print("clean baseline :", run(same_visit_once, baseline))
print("messy_export   :", run(same_visit_once, messy)[:2])

clean baseline : (True, 0, [])
messy_export   : (False, 30)


In [6]:
# The old rule (unique visit_id) vs the new one on the same file
print("visit_id unique  :", run(gxe.ExpectColumnValuesToBeUnique(column="visit_id"), messy)[:2])
print("exact duplicate rows in the file:", messy.duplicated().sum())

visit_id unique  : (False, 30)
exact duplicate rows in the file: 15


In [7]:
# Now plant 6 harder duplicates: same child/day/facility, but a NEW visit_id
planted = messy.sample(6, random_state=1).copy()
planted["visit_id"] = [f"V202501-X{i:03d}" for i in range(len(planted))]
messy_plus = pd.concat([messy, planted], ignore_index=True)

print("visit_id unique  :", run(gxe.ExpectColumnValuesToBeUnique(column="visit_id"), messy_plus)[:2])
print("new compound rule:", run(same_visit_once, messy_plus)[:2])

visit_id unique  : (False, 30)
new compound rule: (False, 42)


In [8]:
# Add it to a COPY of the suite under another name (never touching emr_monthly_extract).
# Build from build_suite.expectations() so we do not write into the stored suite (README quirk).
new_rules = list(build_suite.expectations()) + [same_visit_once]
suite_v2 = ctx.suites.add(gx.ExpectationSuite(name="emr_monthly_extract_v2"))
for e in new_rules:
    suite_v2.add_expectation(e)
vd2 = ctx.validation_definitions.add(gx.ValidationDefinition(name="emr_monthly_v2", data=batch_def, suite=suite_v2))
gate_v2 = ctx.checkpoints.add(gx.Checkpoint(name="emr_gate_v2", validation_definitions=[vd2]))

p = TMP / "data/messy_plus/emr_2025-01.csv"; p.parent.mkdir(parents=True, exist_ok=True)
messy_plus.to_csv(p, index=False)
rep = validate.validate_file(gate_v2, p)
print(len(suite_v2.expectations), "expectations in the extended copy; stored suite still has",
      len(ctx.suites.get("emr_monthly_extract").expectations))
show(rep)

43 expectations in the extended copy; stored suite still has 42
FAIL  messy_plus/emr_2025-01.csv  (2021 rows, 43 expectations)
   [critical] expect_compound_columns_to_be_unique(table): 42 rows (2.1%), e.g. [{'child_id': 'C4793429', 'visit_date': '2025-01-02', 'facility_id': 'SI-HC3'}, {'child_id': 'C1027695', 'visit_date': '2025-01-02', 'facility_id': 'SN-HC4'}, {'child_id': 'C9884080', 'visit_date': '2025-01-04', 'facility_id': 'SN-HC5'}]
   [critical] expect_column_values_to_be_unique(visit_id): 30 rows (1.5%), e.g. ['V202501-00197', 'V202501-00201', 'V202501-00353']
   [critical] expect_column_values_to_be_in_set(sex): 42 rows (2.1%), e.g. ['male', 'male', 'female']
   [critical] expect_column_values_to_be_between(age_months): 12 rows (0.6%), e.g. [70, 69, 68]
   [critical] expect_column_values_to_be_between(muac_cm): 405 rows (23.8%), e.g. [131.0, 131.0, 123.0]
   [critical] expect_column_min_to_be_between(whz): observed -3.0
   [warning] expect_column_values_to_be_null(muac_cm): 

**Answer**

```python
gxe.ExpectCompoundColumnsToBeUnique(column_list=["child_id", "visit_date", "facility_id"])
```

- On the clean baseline it passes (0 rows). On `messy_export` it flags 30 rows: the 15 duplicated visits, counting both copies of each.
- The old `visit_id` rule flags the same 30 rows, because the planted duplicates are exact copies with the same `visit_id`.
- After we plant 6 more copies under **new** `visit_id`s, `visit_id` uniqueness still says 30 (it misses them), while the compound rule says 42 (30 + 6 pairs).
- It was added to a copy of the suite (`emr_monthly_extract_v2`, 43 expectations), built from `build_suite.expectations()` so the stored 42-expectation suite was never touched. To keep it for real, add one `yield` line to `build_suite.py` and rerun it.

Severity: critical is reasonable, as duplicates double-count a child in training. One caution: a child legitimately seen twice in a day at one facility would also be flagged, so check with the clinical team that the rule matches how the clinic works.

## Exercise 3

Create a second suite, `emr_scoring_batch`, for data arriving at
prediction time: no label column, and the row-count bounds suited to a daily batch.

In [9]:
# One day of the baseline is about 65 rows (range 46 to 92 over the month)
per_day = baseline.groupby("visit_date").size()
print(per_day.describe()[["min", "mean", "max"]].round(0).to_dict())
print("facilities per day:", baseline.groupby("visit_date")["facility_id"].nunique().agg(["min", "max"]).to_dict())

{'min': 46.0, 'mean': 65.0, 'max': 92.0}
facilities per day: {'min': 22, 'max': 36}


In [10]:
# Start from the training rules and change only what a daily scoring batch needs
LABEL = "gam_within_90d"

def scoring_expectations():
    for e in build_suite.expectations():
        kw = e.configuration.kwargs
        if kw.get("column") == LABEL:
            continue                                          # label does not exist at prediction time
        if e.expectation_type == "expect_table_columns_to_match_set":
            yield gxe.ExpectTableColumnsToMatchSet(
                column_set=[c for c in build_suite.COLUMNS if c != LABEL], exact_match=True)
        elif e.expectation_type == "expect_table_row_count_to_be_between":
            yield gxe.ExpectTableRowCountToBeBetween(min_value=20, max_value=300)    # a day, not a month
        elif e.expectation_type == "expect_column_unique_value_count_to_be_between":
            yield gxe.ExpectColumnUniqueValueCountToBeBetween(
                column="facility_id", min_value=10, max_value=None, severity="warning")  # a day, not a month
        else:
            yield e

rules = list(scoring_expectations())
print(len(rules), "expectations (training suite has", len(list(build_suite.expectations())), ")")

39 expectations (training suite has 42 )


In [11]:
# Save as a new suite in the temp project, with its own validation definition and checkpoint
suite_s = ctx.suites.add(gx.ExpectationSuite(name="emr_scoring_batch"))
for e in rules:
    suite_s.add_expectation(e)
vd_s = ctx.validation_definitions.add(gx.ValidationDefinition(name="emr_scoring", data=batch_def, suite=suite_s))
gate_s = ctx.checkpoints.add(gx.Checkpoint(name="scoring_gate", validation_definitions=[vd_s]))

def save(df, name):
    p = TMP / f"data/{name}/emr_2025-01-15.csv"; p.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(p, index=False); return p

In [12]:
# A good daily batch: one day of the baseline, label column removed
day = baseline[baseline["visit_date"] == "2025-01-15"].drop(columns=[LABEL])
print("label-free daily batch:", day.shape)
show(validate.validate_file(gate_s, save(day, "scoring_ok")))

# Why a second suite is needed: the training suite rejects the same batch
print()
show(validate.validate_file(gate, save(day, "scoring_ok_vs_training")))

label-free daily batch: (72, 22)


PASS  scoring_ok/emr_2025-01-15.csv  (72 rows, 39 expectations)



FAIL  scoring_ok_vs_training/emr_2025-01-15.csv  (72 rows, 42 expectations)
   [critical] expect_column_values_to_be_in_set(gam_within_90d): 
   [critical] expect_column_values_to_not_be_null(gam_within_90d): 
   [critical] expect_table_columns_to_match_set(table): observed ['age_months', 'birth_size', 'child_id', 'currently_breastfed', 'facility_id', 'height_cm', 'household_size', 'mother_age', 'mother_education', 'muac_cm', 'prior_sam_admission', 'recent_diarrhea', 'recent_fever', 'region', 'residence', 'sex', 'vaccinated_for_age', 'visit_date', 'visit_id', 'water_source_minutes', 'weight_kg', 'whz']
   [critical] expect_table_row_count_to_be_between(table): observed 72
   [warning] expect_column_mean_to_be_between(gam_within_90d): 


In [13]:
# A broken daily batch: MUAC in mm, a misspelt region, a typo'd sex code
bad = day.copy()
bad["muac_cm"] = bad["muac_cm"] * 10
bad.loc[bad.index[:3], "region"] = "Oromiya"
bad.loc[bad.index[3:5], "sex"] = "female"
show(validate.validate_file(gate_s, save(bad, "scoring_bad")))

# And a half-finished export: only 8 rows
print()
show(validate.validate_file(gate_s, save(day.head(8), "scoring_short")))

FAIL  scoring_bad/emr_2025-01-15.csv  (72 rows, 39 expectations)
   [critical] expect_column_values_to_be_in_set(region): 3 rows (4.2%), e.g. ['Oromiya', 'Oromiya', 'Oromiya']
   [critical] expect_column_values_to_be_in_set(sex): 2 rows (2.8%), e.g. ['female', 'female']
   [critical] expect_column_values_to_be_between(muac_cm): 51 rows (100.0%), e.g. [148.0, 129.0, 135.0]



FAIL  scoring_short/emr_2025-01-15.csv  (8 rows, 39 expectations)
   [critical] expect_table_row_count_to_be_between(table): observed 8
   [warning] expect_column_unique_value_count_to_be_between(facility_id): observed 8
   [warning] expect_column_median_to_be_between(age_months): observed 3.0


**Answer**

`emr_scoring_batch` is the training rules with three changes, 39 expectations in all:
1. The label `gam_within_90d` is gone: its checks are dropped and the exact column set no longer lists it.
2. Row count is 20 to 300 (a day here is 46 to 92 rows; the monthly bound of 500 to 5000 would reject every day).
3. The distinct-facility warning is 10 or more instead of 20 (days show 22 to 36 facilities).

Results: a label-free day (72 rows) **passes**. The same batch **fails the training suite** (label missing, only 72 rows), which is why a second suite is needed. A broken day fails on `region` ("Oromiya"), `sex` ("female") and `muac_cm` in mm, and an 8-row export fails on the row-count bound. The unit, code and cohort checks carry over unchanged, and they matter more at scoring time, because a bad value goes straight into a live prediction.

## Exercise 4

Make the `messy_export` month pass by *fixing the data*, not by loosening
the rules: write a small `clean.py` that drops duplicates, maps
"female"/"male", converts MUAC > 50 from mm to cm, and removes out-of-cohort
rows. Should a cleaning step like this live in the pipeline, or should the
EMR export be fixed upstream?

In [14]:
# Before: the real suite, the real gate, on the untouched messy_export file
before = validate.validate_file(gate, TMP / "data/messy_export/emr_2025-01.csv")
show(before)

FAIL  messy_export/emr_2025-01.csv  (2015 rows, 42 expectations)
   [critical] expect_column_values_to_be_unique(visit_id): 30 rows (1.5%), e.g. ['V202501-00197', 'V202501-00201', 'V202501-00353']
   [critical] expect_column_values_to_be_in_set(sex): 42 rows (2.1%), e.g. ['male', 'male', 'female']
   [critical] expect_column_values_to_be_between(age_months): 12 rows (0.6%), e.g. [70, 69, 68]
   [critical] expect_column_values_to_be_between(muac_cm): 404 rows (23.8%), e.g. [131.0, 131.0, 123.0]
   [critical] expect_column_min_to_be_between(whz): observed -3.0
   [warning] expect_column_values_to_be_null(muac_cm): 79 rows (20.1%), e.g. [131.0, 123.0, 133.0]


In [15]:
# The cleaning step (this is what would go in clean.py). It returns the data and a log of what it changed.
def clean(df):
    log = {"rows_in": len(df)}
    df = df.copy()

    # 1. exact duplicates, then the same child/day/facility under another visit_id
    df = df.drop_duplicates()
    df = df.drop_duplicates(subset=["child_id", "visit_date", "facility_id"], keep="first")
    log["duplicates_dropped"] = log["rows_in"] - len(df)

    # 2. free-text sex codes
    log["sex_mapped"] = int(df["sex"].isin(["female", "male"]).sum())
    df["sex"] = df["sex"].replace({"female": "F", "male": "M"})

    # 3. MUAC recorded in mm (nothing real is above 50 cm)
    mm = df["muac_cm"] > 50
    log["muac_mm_to_cm"] = int(mm.sum())
    df.loc[mm, "muac_cm"] = df.loc[mm, "muac_cm"] / 10

    # 4. protocol: no MUAC under 6 months (the Amhara update recorded it anyway)
    infants = (df["age_months"] < 6) & df["muac_cm"].notna()
    log["infant_muac_blanked"] = int(infants.sum())
    df.loc[infants, "muac_cm"] = float("nan")

    # 5. out-of-cohort rows: aged out of the programme, or already malnourished (referral, not scoring)
    out = (df["age_months"] > 59) | (df["whz"] < -2) | (df["muac_cm"] < 12.5)
    log["out_of_cohort_removed"] = int(out.sum())
    df = df[~out]

    log["rows_out"] = len(df)
    return df.reset_index(drop=True), log

cleaned, log = clean(messy)
print(json.dumps(log, indent=1))

{
 "rows_in": 2015,
 "duplicates_dropped": 15,
 "sex_mapped": 40,
 "muac_mm_to_cm": 402,
 "infant_muac_blanked": 78,
 "out_of_cohort_removed": 37,
 "rows_out": 1963
}


In [16]:
# After: save the cleaned file and run it through the same gate
out = TMP / "data/messy_cleaned/emr_2025-01.csv"; out.parent.mkdir(parents=True, exist_ok=True)
cleaned.to_csv(out, index=False)
after = validate.validate_file(gate, out)
show(after)
print("gate:", "BLOCKED" if after["critical_failures"] else "OK: no critical failures")

PASS  messy_cleaned/emr_2025-01.csv  (1963 rows, 42 expectations)
gate: OK: no critical failures


In [17]:
# Side by side
pd.DataFrame({"before": [before["rows"], len(before["critical_failures"]), len(before["warnings"])],
              "after":  [after["rows"],  len(after["critical_failures"]),  len(after["warnings"])]},
             index=["rows", "critical failures", "warnings"])

,before,after
rows,2015,1963
critical failures,5,0
warnings,1,0


**Answer**

The cleaning function (it would sit in `clean.py` as is) removed 15 duplicates, mapped 40 sex codes, converted 402 MUAC values from mm to cm, blanked 78 infant MUAC values, and dropped 37 out-of-cohort rows. Before: **FAIL**, 5 critical failures and 1 warning. After: **PASS**, 1963 rows, 0 critical, 0 warnings, through the repo's own `validate_file` and the `emr_gate` checkpoint, with the rules untouched. The infant step is not in the exercise list; it clears the one warning, and the protocol says infants get no MUAC.

**Pipeline or upstream?** Fix it upstream, and use the cleaning step only as a temporary bridge:
- The cleaner is a guess. "MUAC above 50 means mm" is right today, but it hides a real EMR bug (the Amhara update), and the next unit change will not look like this one.
- It quietly changes the data. Dropping rows changes who the model sees; here 52 of 2015 rows.
- The gate is only worth something if it fails loudly. If the pipeline fixes everything, nobody tells the EMR team.

If it has to live in the pipeline: keep the raw file, clean into a new file, log the counts (as `log` does), alert the data owner whenever any count is above zero, and run the gate on the raw file as well as the cleaned one. Duplicates are the one fix that is safe to automate for good, since they are not a judgement about values.